In [3]:
import pandas as pd
import numpy as np
from enum import Enum

# ==========================================
# 1. ENUMS & CONFIGURATION
# ==========================================

class TargetBrands(str, Enum):
    STAART = 'Staart'

class ProductStages(str, Enum):
    STAGE_1 = 'Staart Prime (Stage 1)'
    STAGES_2_3_4 = 'Staart Prime (Stages 2,3,4)'
    OTHER = 'Other'

class ProductWeights(str, Enum):
    WEIGHT_200G = '200g'
    WEIGHT_400G = '400g'
    OTHER = 'Other'

class Metrics(str, Enum):
    QUANTITY = 'Qty'
    REVENUE = 'Revenue'

class SheetNames(str, Enum):
    STATE_ANALYSIS = 'State_Analysis'
    MONTHLY_PLATFORM_ANALYTICS = 'Monthly_Platform_Analytics'
    MOM_SALES_TREND = 'MoM_Sales_Trend'

# Configuration mapping for easy global changes
CONFIG = {
    "INPUT_FILE_PATH": "Logistics Dashboard - Unified_Sales.csv",
    "OUTPUT_FILE_PATH": "Staart_Prime_Comprehensive_Report.xlsx",
    "TARGET_PLATFORMS": ['Amazon', 'Flipkart', '7nshop', 'TATA', 'Myntra'],
    "DATE_FORMAT": '%d-%m-%y'
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def map_product_stage(product_name: str) -> str:
    """Maps the raw product name to its respective Staart Prime Stage."""
    if 'Stage 1' in str(product_name):
        return ProductStages.STAGE_1.value
    if 'Stages 2-3-4' in str(product_name):
        return ProductStages.STAGES_2_3_4.value
    return ProductStages.OTHER.value

def map_product_weight(product_name: str) -> str:
    """Maps the raw product name to its standardized weight."""
    if '200 g' in str(product_name):
        return ProductWeights.WEIGHT_200G.value
    if '400 g' in str(product_name):
        return ProductWeights.WEIGHT_400G.value
    return ProductWeights.OTHER.value

def load_and_preprocess_data() -> pd.DataFrame:
    """Loads the CSV and applies initial filtering, mapping, and date parsing."""
    df_raw = pd.read_csv(CONFIG["INPUT_FILE_PATH"])

    is_target_brand = df_raw['Brand'] == TargetBrands.STAART.value
    is_target_platform = df_raw['Platform'].isin(CONFIG["TARGET_PLATFORMS"])
    df_filtered = df_raw[is_target_brand & is_target_platform].copy()

    # Parse dates and setup chronological month periods
    df_filtered['Date_parsed'] = pd.to_datetime(df_filtered['Date'], format=CONFIG["DATE_FORMAT"], errors='coerce')
    df_filtered['Month_Name_Year'] = df_filtered['Date_parsed'].dt.strftime('%b-%Y')
    df_filtered['Month_Period'] = df_filtered['Date_parsed'].dt.to_period('M')

    # Standardize data columns
    df_filtered['Stage'] = df_filtered['Product Name'].apply(map_product_stage)
    df_filtered['Weight'] = df_filtered['Product Name'].apply(map_product_weight)
    df_filtered['State'] = df_filtered['State'].str.title()

    return df_filtered

# ==========================================
# 3. REPORT GENERATORS
# ==========================================

def generate_state_analysis(df: pd.DataFrame) -> dict:
    """Report 1: State-wise REVENUE ONLY breakdown, split into 3 separate tables per SKU."""

    ordered_periods = sorted(df['Month_Period'].dropna().unique())
    ordered_months = [p.strftime('%b-%Y') for p in ordered_periods]

    grouped_df = df.groupby(['State', 'Month_Name_Year', 'Stage', 'Weight'])[Metrics.REVENUE.value].sum().reset_index()

    tables = {}
    stage_weight_combinations = [
        (ProductStages.STAGE_1.value, ProductWeights.WEIGHT_200G.value),
        (ProductStages.STAGE_1.value, ProductWeights.WEIGHT_400G.value),
        (ProductStages.STAGES_2_3_4.value, ProductWeights.WEIGHT_400G.value)
    ]

    all_active_states = sorted(df['State'].dropna().unique())

    for stage, weight in stage_weight_combinations:
        sub_df = grouped_df[(grouped_df['Stage'] == stage) & (grouped_df['Weight'] == weight)]

        pivot_df = sub_df.pivot_table(
            index='State',
            columns='Month_Name_Year',
            values=Metrics.REVENUE.value,
            aggfunc='sum',
            fill_value=0
        )

        # Ensure uniform rows and columns
        for state in all_active_states:
            if state not in pivot_df.index:
                pivot_df.loc[state] = 0
        for month in ordered_months:
            if month not in pivot_df.columns:
                pivot_df[month] = 0

        pivot_df = pivot_df[ordered_months]

        # Subtotals and Contribution
        num_months = len(ordered_months)
        pivot_df['Avg Revenue / Month'] = pivot_df[ordered_months].sum(axis=1) / num_months

        total_avg = pivot_df['Avg Revenue / Month'].sum()
        if total_avg > 0:
            pivot_df['State Contribution (%)'] = (pivot_df['Avg Revenue / Month'] / total_avg) * 100
        else:
            pivot_df['State Contribution (%)'] = 0

        pivot_df = pivot_df.sort_index()
        table_title = f"{stage} ({weight}) - State Revenue Trends"
        pivot_df.index.name = table_title

        tables[table_title] = pivot_df.round(2)

    return tables

def generate_monthly_platform_analytics(df: pd.DataFrame) -> dict:
    """Report 2: Platform-wise breakdown, separating Revenue and Quantity into distinct tables per SKU."""

    ordered_periods = sorted(df['Month_Period'].dropna().unique())
    ordered_months = [p.strftime('%b-%Y') for p in ordered_periods]

    # We now fetch BOTH Quantity and Revenue for this report
    grouped_df = df.groupby(['Platform', 'Month_Name_Year', 'Stage', 'Weight'])[[Metrics.REVENUE.value, Metrics.QUANTITY.value]].sum().reset_index()

    tables = {}
    stage_weight_combinations = [
        (ProductStages.STAGE_1.value, ProductWeights.WEIGHT_200G.value),
        (ProductStages.STAGE_1.value, ProductWeights.WEIGHT_400G.value),
        (ProductStages.STAGES_2_3_4.value, ProductWeights.WEIGHT_400G.value)
    ]

    for stage, weight in stage_weight_combinations:
        sub_df = grouped_df[(grouped_df['Stage'] == stage) & (grouped_df['Weight'] == weight)]

        # Loop through each metric to generate separate tables (Revenue first, then Quantity)
        for metric in [Metrics.REVENUE.value, Metrics.QUANTITY.value]:
            pivot_df = sub_df.pivot_table(
                index='Platform',
                columns='Month_Name_Year',
                values=metric,
                aggfunc='sum',
                fill_value=0
            )

            # Ensure uniform rows and columns
            for platform in CONFIG["TARGET_PLATFORMS"]:
                if platform not in pivot_df.index:
                    pivot_df.loc[platform] = 0
            for month in ordered_months:
                if month not in pivot_df.columns:
                    pivot_df[month] = 0

            pivot_df = pivot_df[ordered_months]

            # Subtotals and Contribution
            num_months = len(ordered_months)
            avg_col_name = f'Avg {metric} / Month'
            pivot_df[avg_col_name] = pivot_df[ordered_months].sum(axis=1) / num_months

            total_avg = pivot_df[avg_col_name].sum()
            if total_avg > 0:
                pivot_df['Platform Contribution (%)'] = (pivot_df[avg_col_name] / total_avg) * 100
            else:
                pivot_df['Platform Contribution (%)'] = 0

            table_title = f"{stage} ({weight}) - Platform {metric} Trends"
            pivot_df.index.name = table_title

            tables[table_title] = pivot_df.round(2)

    return tables

def generate_mom_sales_trend(df: pd.DataFrame) -> dict:
    """Report 3: Platform-wise REVENUE ONLY Trend & % Growth, split into 3 separate tables per SKU."""

    grouped_df = df.groupby(['Platform', 'Month_Period', 'Stage', 'Weight'])[Metrics.REVENUE.value].sum().reset_index()

    tables = {}
    stage_weight_combinations = [
        (ProductStages.STAGE_1.value, ProductWeights.WEIGHT_200G.value),
        (ProductStages.STAGE_1.value, ProductWeights.WEIGHT_400G.value),
        (ProductStages.STAGES_2_3_4.value, ProductWeights.WEIGHT_400G.value)
    ]

    for stage, weight in stage_weight_combinations:
        sub_df = grouped_df[(grouped_df['Stage'] == stage) & (grouped_df['Weight'] == weight)].copy()

        # Sort chronologically to ensure percentage change calculates correctly
        sub_df = sub_df.sort_values(['Platform', 'Month_Period'])

        # Calculate Month-on-Month Revenue Growth (%)
        sub_df['MoM Rev Growth (%)'] = sub_df.groupby('Platform')[Metrics.REVENUE.value].pct_change() * 100

        # Format the month string for the final output
        sub_df['Month'] = sub_df['Month_Period'].dt.strftime('%b-%Y')

        # Reorder and select final columns
        sub_df = sub_df[['Platform', 'Month', Metrics.REVENUE.value, 'MoM Rev Growth (%)']]

        table_title = f"{stage} ({weight}) - MoM Revenue Trend"

        # We fill NaNs with '-' so the first month doesn't show an ugly "NaN" for growth
        tables[table_title] = sub_df.round(2).fillna('-')

    return tables

# ==========================================
# 4. MAIN EXECUTION & EXPORT
# ==========================================

def main():
    print(f"Loading data from {CONFIG['INPUT_FILE_PATH']}...")
    df_cleaned = load_and_preprocess_data()

    print("Generating Comprehensive Product-Level Reports...")
    report_1_tables_dict = generate_state_analysis(df_cleaned)
    report_2_tables_dict = generate_monthly_platform_analytics(df_cleaned)
    report_3_tables_dict = generate_mom_sales_trend(df_cleaned)

    print(f"Saving compiled reports to {CONFIG['OUTPUT_FILE_PATH']}...")
    with pd.ExcelWriter(CONFIG["OUTPUT_FILE_PATH"], engine='openpyxl') as writer:

        # Write Report 1 (State Analysis)
        start_row = 0
        for title, df_table in report_1_tables_dict.items():
            df_table.to_excel(writer, sheet_name=SheetNames.STATE_ANALYSIS.value, startrow=start_row)
            start_row += len(df_table.index) + 4  # Add spacing

        # Write Report 2 (Monthly Platform Analytics - Now writes 6 tables)
        start_row = 0
        for title, df_table in report_2_tables_dict.items():
            df_table.to_excel(writer, sheet_name=SheetNames.MONTHLY_PLATFORM_ANALYTICS.value, startrow=start_row)
            start_row += len(df_table.index) + 4

        # Write Report 3 (MoM Revenue Trend)
        start_row = 0
        for title, df_table in report_3_tables_dict.items():
            # Adding the title as a standalone row since this one isn't a pivot table index name
            worksheet = writer.book.create_sheet(SheetNames.MOM_SALES_TREND.value) if start_row == 0 else writer.sheets[SheetNames.MOM_SALES_TREND.value]
            worksheet.cell(row=start_row+1, column=1, value=title)

            # Writing the flat table below the title
            df_table.to_excel(writer, sheet_name=SheetNames.MOM_SALES_TREND.value, startrow=start_row+1, index=False)
            start_row += len(df_table.index) + 5

    print("Export Complete! Your comprehensive SKU-level Excel report is ready.")

if __name__ == "__main__":
    main()

Loading data from Logistics Dashboard - Unified_Sales.csv...


FileNotFoundError: [Errno 2] No such file or directory: 'Logistics Dashboard - Unified_Sales.csv'

In [14]:
import pandas as pd
import numpy as np
from enum import Enum

# ==========================================
# 1. ENUMS & CONFIGURATION
# ==========================================

class TargetBrands(str, Enum):
    STAART = 'Staart'

class Metrics(str, Enum):
    QUANTITY = 'Qty'
    REVENUE = 'Revenue'

class SheetNames(str, Enum):
    STATE_ANALYSIS = 'State_Analysis'
    MONTHLY_PLATFORM_ANALYTICS = 'Monthly_Platform_Analytics'
    MOM_SALES_TREND = 'MoM_Sales_Trend'

# Configuration mapping
CONFIG = {
    "INPUT_FILE_PATH": "Logistics Dashboard - Unified_Sales.csv",
    "OUTPUT_FILE_PATH": "Staart_Overall_Brand_Report.xlsx",
    "TARGET_PLATFORMS": ['Amazon', 'Flipkart', '7nshop', 'TATA', 'Myntra'],
    "DATE_FORMAT": '%d-%m-%y'
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def load_and_preprocess_data() -> pd.DataFrame:
    """Loads the CSV, filters for Staart brand, and prepares date columns."""
    df_raw = pd.read_csv(CONFIG["INPUT_FILE_PATH"])

    # Filter for Staart Brand and Target Platforms
    is_target_brand = df_raw['Brand'] == TargetBrands.STAART.value
    is_target_platform = df_raw['Platform'].isin(CONFIG["TARGET_PLATFORMS"])
    df_filtered = df_raw[is_target_brand & is_target_platform].copy()

    # Parse dates and setup chronological month periods
    df_filtered['Date_parsed'] = pd.to_datetime(df_filtered['Date'], format=CONFIG["DATE_FORMAT"], errors='coerce')
    df_filtered['Month_Name_Year'] = df_filtered['Date_parsed'].dt.strftime('%b-%Y')
    df_filtered['Month_Period'] = df_filtered['Date_parsed'].dt.to_period('M')

    # Standardize State Names
    df_filtered['State'] = df_filtered['State'].str.title()

    return df_filtered

# ==========================================
# 3. REPORT GENERATORS
# ==========================================

def generate_state_analysis(df: pd.DataFrame) -> pd.DataFrame:
    """Report 1: State-wise Overall REVENUE Trends for the entire brand."""

    ordered_periods = sorted(df['Month_Period'].dropna().unique())
    ordered_months = [p.strftime('%b-%Y') for p in ordered_periods]

    # Group by State and Month (Aggregating all products together)
    grouped_df = df.groupby(['State', 'Month_Name_Year'])[Metrics.REVENUE.value].sum().reset_index()

    pivot_df = grouped_df.pivot_table(
        index='State',
        columns='Month_Name_Year',
        values=Metrics.REVENUE.value,
        aggfunc='sum',
        fill_value=0
    )

    all_active_states = sorted(df['State'].dropna().unique())

    # Ensure uniform rows and columns
    for state in all_active_states:
        if state not in pivot_df.index:
            pivot_df.loc[state] = 0
    for month in ordered_months:
        if month not in pivot_df.columns:
            pivot_df[month] = 0

    pivot_df = pivot_df[ordered_months]

    # Subtotals and Contribution
    num_months = len(ordered_months)
    pivot_df['Avg Revenue / Month'] = pivot_df[ordered_months].sum(axis=1) / num_months

    total_avg = pivot_df['Avg Revenue / Month'].sum()
    if total_avg > 0:
        pivot_df['State Contribution (%)'] = (pivot_df['Avg Revenue / Month'] / total_avg) * 100
    else:
        pivot_df['State Contribution (%)'] = 0

    pivot_df = pivot_df.sort_index()
    pivot_df.index.name = "Overall Staart Brand - State Revenue Trends"

    return pivot_df.round(2)

def generate_monthly_platform_analytics(df: pd.DataFrame) -> pd.DataFrame:
    """Report 2: Platform-wise Overall REVENUE Trends for the entire brand."""

    ordered_periods = sorted(df['Month_Period'].dropna().unique())
    ordered_months = [p.strftime('%b-%Y') for p in ordered_periods]

    # Group by Platform and Month
    grouped_df = df.groupby(['Platform', 'Month_Name_Year'])[Metrics.REVENUE.value].sum().reset_index()

    pivot_df = grouped_df.pivot_table(
        index='Platform',
        columns='Month_Name_Year',
        values=Metrics.REVENUE.value,
        aggfunc='sum',
        fill_value=0
    )

    # Ensure uniform rows and columns
    for platform in CONFIG["TARGET_PLATFORMS"]:
        if platform not in pivot_df.index:
            pivot_df.loc[platform] = 0
    for month in ordered_months:
        if month not in pivot_df.columns:
            pivot_df[month] = 0

    pivot_df = pivot_df[ordered_months]

    # Subtotals and Contribution
    num_months = len(ordered_months)
    pivot_df['Avg Revenue / Month'] = pivot_df[ordered_months].sum(axis=1) / num_months

    total_avg = pivot_df['Avg Revenue / Month'].sum()
    if total_avg > 0:
        pivot_df['Platform Contribution (%)'] = (pivot_df['Avg Revenue / Month'] / total_avg) * 100
    else:
        pivot_df['Platform Contribution (%)'] = 0

    pivot_df.index.name = "Overall Staart Brand - Platform Revenue Trends"

    return pivot_df.round(2)

def generate_mom_sales_trend(df: pd.DataFrame) -> pd.DataFrame:
    """Report 3: Platform-wise Overall QUANTITY Trends for the entire brand."""

    ordered_periods = sorted(df['Month_Period'].dropna().unique())
    ordered_months = [p.strftime('%b-%Y') for p in ordered_periods]

    # Group by Platform and Month for Quantity
    grouped_df = df.groupby(['Platform', 'Month_Name_Year'])[Metrics.QUANTITY.value].sum().reset_index()

    pivot_df = grouped_df.pivot_table(
        index='Platform',
        columns='Month_Name_Year',
        values=Metrics.QUANTITY.value,
        aggfunc='sum',
        fill_value=0
    )

    # Ensure uniform rows and columns
    for platform in CONFIG["TARGET_PLATFORMS"]:
        if platform not in pivot_df.index:
            pivot_df.loc[platform] = 0
    for month in ordered_months:
        if month not in pivot_df.columns:
            pivot_df[month] = 0

    pivot_df = pivot_df[ordered_months]

    # Subtotals and Contribution
    num_months = len(ordered_months)
    pivot_df['Avg Qty / Month'] = pivot_df[ordered_months].sum(axis=1) / num_months

    total_avg = pivot_df['Avg Qty / Month'].sum()
    if total_avg > 0:
        pivot_df['Platform Contribution (%)'] = (pivot_df['Avg Qty / Month'] / total_avg) * 100
    else:
        pivot_df['Platform Contribution (%)'] = 0

    pivot_df.index.name = "Overall Staart Brand - MoM Quantity Trend"

    return pivot_df.round(2)

# ==========================================
# 4. MAIN EXECUTION & EXPORT
# ==========================================

def main():
    print(f"Loading data from {CONFIG['INPUT_FILE_PATH']}...")
    df_cleaned = load_and_preprocess_data()

    print("Generating Brand-Level Reports...")
    report_1 = generate_state_analysis(df_cleaned)
    report_2 = generate_monthly_platform_analytics(df_cleaned)
    report_3 = generate_mom_sales_trend(df_cleaned)

    print(f"Saving compiled reports to {CONFIG['OUTPUT_FILE_PATH']}...")
    with pd.ExcelWriter(CONFIG["OUTPUT_FILE_PATH"], engine='openpyxl') as writer:

        # Write Report 1 (State Analysis)
        report_1.to_excel(writer, sheet_name=SheetNames.STATE_ANALYSIS.value)

        # Write Report 2 (Monthly Platform Revenue)
        report_2.to_excel(writer, sheet_name=SheetNames.MONTHLY_PLATFORM_ANALYTICS.value)

        # Write Report 3 (MoM Quantity Trend)
        report_3.to_excel(writer, sheet_name=SheetNames.MOM_SALES_TREND.value)

    print("Export Complete! Your brand-level Excel report is ready.")

if __name__ == "__main__":
    main()

Loading data from Logistics Dashboard - Unified_Sales.csv...
Generating Brand-Level Reports...
Saving compiled reports to Staart_Overall_Brand_Report.xlsx...
Export Complete! Your brand-level Excel report is ready.
